In [ ]:
!pip install "datasets<3.0.0" soundfile transformers accelerate torch resemblyzer librosa pandas scipy

In [ ]:
import torch
import numpy as np
import pandas as pd
import soundfile as sf
import os
from transformers import VitsModel, AutoTokenizer
from resemblyzer import VoiceEncoder, preprocess_wav

# ── Config ─────────────────────────────────────────────────────
MODEL_NAME = "facebook/mms-tts-urd-script_arabic"

# ── Load TTS model + speaker encoder ─────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = VitsModel.from_pretrained(MODEL_NAME)
model.eval()
encoder = VoiceEncoder()

def run_evaluation(dataset_name, samples_list, text_key, limit):
    """Generate TTS and compute Resemblyzer similarities without anchor audios."""
    base_dir = f"mushra_{dataset_name}"
    # Removed 'anchor' subfolder
    for sub in ["reference", "generated"]:
        os.makedirs(f"{base_dir}/{sub}", exist_ok=True)

    results = []
    for i, sample in enumerate(samples_list[:limit]):
        try:
            text = sample[text_key]
            ref_array = np.array(sample["audio"]["array"]).astype(np.float32)
            ref_sr = sample["audio"]["sampling_rate"]
            if ref_array.ndim > 1:
                ref_array = ref_array.mean(axis=1)

            ref_path = f"{base_dir}/reference/sample_{i}.wav"
            gen_path = f"{base_dir}/generated/sample_{i}.wav"

            # 1. Reference
            sf.write(ref_path, ref_array, ref_sr)

            # 2. TTS generation
            inputs = tokenizer(text, return_tensors="pt")
            with torch.no_grad():
                output = model(**inputs)
            gen_speech = output.waveform[0].cpu().numpy()
            gen_sr = model.config.sampling_rate
            sf.write(gen_path, gen_speech, gen_sr)

            # 3. Resemblyzer embeddings
            ref_wav = preprocess_wav(ref_path)
            gen_wav = preprocess_wav(gen_path)
            ref_emb = encoder.embed_utterance(ref_wav)
            gen_emb = encoder.embed_utterance(gen_wav)

            ref_gen_sim = float(np.dot(ref_emb, gen_emb))

            results.append({
                "sample_id": i,
                "urdu_text": text,
                "gender": sample.get("gender", "N/A"),
                "audio_duration_sec": round(len(ref_array) / ref_sr, 2),
                "ref_gen_similarity": round(ref_gen_sim, 4),
            })
            print(f"  [{i+1:02d}/{limit}] sim={ref_gen_sim:.4f} gender={sample.get('gender')} text={text[:30]}...")
        except Exception as e:
            print(f"  ☑ Sample {i} failed: {e}")

    print(f"\n✅ {dataset_name}: {len(results)}/{limit} samples evaluated")
    return results

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/302 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Loaded the voice encoder model on cuda in 0.51 seconds.


In [ ]:
from google.colab import userdata
try:
    hf_token = userdata.get("HF_TOKEN")
    print("✅ HF token loaded from Colab secrets")
except Exception:
    hf_token = None
    print("⚠️  HF_TOKEN not found in secrets — proceeding without token (may fail for gated datasets)")


✅ HF token loaded from Colab secrets


In [ ]:
from datasets import load_dataset

# ── 3. Literary / Storytelling style (60 samples, Male, Urdu) ──
print("‹‹ Loading Literary (BOOK) Male Samples (Goal: 60) ...")

ds_book_stream = load_dataset(
    "humairawan/UrduSpeech",
    data_dir="Urdu",
    split="train",
    token=hf_token,
    streaming=True,
)

book_male_samples = []
count = 0

for sample in ds_book_stream:
    count += 1
    style = str(sample.get("style", "")).strip().upper()
    gender = str(sample.get("gender", "")).strip().capitalize()

    if style == "BOOK" and gender == "Male":
        book_male_samples.append(sample)

    # Provide visual feedback every 1000 samples scanned
    if count % 1000 == 0:
        print(f"  Checked {count} samples... Found {len(book_male_samples)} matches so far.")

    if len(book_male_samples) >= 60:
        break

    # Safety break to avoid infinite loops if data is missing
    if count > 50000:
        print("  Stopping search: scanned 50,000 samples without finding 60 matches.")
        break

print(f"  Collected {len(book_male_samples)} BOOK Male samples after scanning {count} total entries.")
results_book = run_evaluation("humair_book_male", book_male_samples, text_key="text", limit=60)

‹‹ Loading Literary (BOOK) Male Samples (Goal: 60) ...


Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

  Checked 1000 samples... Found 0 matches so far.
  Checked 2000 samples... Found 0 matches so far.
  Checked 3000 samples... Found 0 matches so far.
  Checked 4000 samples... Found 0 matches so far.
  Checked 5000 samples... Found 0 matches so far.
  Checked 6000 samples... Found 0 matches so far.
  Checked 7000 samples... Found 0 matches so far.
  Checked 8000 samples... Found 0 matches so far.
  Checked 9000 samples... Found 0 matches so far.
  Checked 10000 samples... Found 0 matches so far.
  Checked 11000 samples... Found 0 matches so far.
  Checked 12000 samples... Found 0 matches so far.
  Checked 13000 samples... Found 21 matches so far.
  Collected 60 BOOK Male samples after scanning 13677 total entries.
  [01/60] sim=0.6175 gender=Male text=یا یہ طلسم ہے کہ اگر پھٹکری او...
  [02/60] sim=0.6194 gender=Male text=میں جان و دل سے اُسے چاہتی تھی...
  [03/60] sim=0.6490 gender=Male text=سب آدمی اپنے اپنے عہدوں پر مست...
  [04/60] sim=0.6012 gender=Male text=تب انہوں نے فرمایا کہ 

In [ ]:
# ── 2. Formal / Wiki (30 samples, Male, Urdu) ──
print("\n‹‹ Loading Formal (WIKI) Male Samples (Goal: 30) ...")
# Re-initializing stream to ensure fresh traversal
ds_wiki_stream = load_dataset(
    "humairawan/UrduSpeech",
    data_dir="Urdu",
    split="train",
    token=hf_token,
    streaming=True,
)

wiki_male_samples = []
count = 0
for sample in ds_wiki_stream:
    count += 1
    style = str(sample.get("style", "")).strip().upper()
    gender = str(sample.get("gender", "")).strip().capitalize()

    if style == "WIKI" and gender == "Male":
        wiki_male_samples.append(sample)

    if count % 1000 == 0:
        print(f"  Checked {count} samples... Found {len(wiki_male_samples)} matches so far.")

    if len(wiki_male_samples) >= 30:
        break

    if count > 50000:
        print("  Stopping search: scanned 50,000 samples.")
        break

print(f"  Collected {len(wiki_male_samples)} WIKI Male samples after scanning {count} total entries.")
results_wiki = run_evaluation("humair_wiki_male", wiki_male_samples, text_key="text", limit=30)


‹‹ Loading Formal (WIKI) Male Samples (Goal: 30) ...


Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

  Checked 1000 samples... Found 0 matches so far.
  Checked 2000 samples... Found 0 matches so far.
  Checked 3000 samples... Found 0 matches so far.
  Checked 4000 samples... Found 0 matches so far.
  Checked 5000 samples... Found 0 matches so far.
  Checked 6000 samples... Found 0 matches so far.
  Checked 7000 samples... Found 0 matches so far.
  Checked 8000 samples... Found 0 matches so far.
  Checked 9000 samples... Found 0 matches so far.
  Checked 10000 samples... Found 0 matches so far.
  Checked 11000 samples... Found 0 matches so far.
  Checked 12000 samples... Found 0 matches so far.
  Collected 30 WIKI Male samples after scanning 12691 total entries.
  [01/30] sim=0.6946 gender=Male text=آشوب چشم کی شناخت آنکھ کی جھلی...
  [02/30] sim=0.6180 gender=Male text=جَبہَۃُ الاَکْراد شام کی خانہ ...
  [03/30] sim=0.6237 gender=Male text=ایسا لگتا ہے کہ یہ نام میدانی ...
  [04/30] sim=0.6092 gender=Male text=اس رکاوٹ سے نمٹنے کے لیے، حکوم...
  [05/30] sim=0.6934 gender=Male text=سر

In [ ]:
# ── 1. Conversational (60 samples, Male, Urdu) ──
print("‹‹ Loading Conversational Male Samples (Goal: 60) ...")
ds_stream = load_dataset(
    "humairawan/UrduSpeech",
    data_dir="Urdu",
    split="train",
    token=hf_token,
    streaming=True,
)

conv_male_samples = []
count = 0
for sample in ds_stream:
    count += 1
    style = str(sample.get("style", "")).strip().upper()
    gender = str(sample.get("gender", "")).strip().capitalize()

    if style == "CONV" and gender == "Male":
        conv_male_samples.append(sample)

    if count % 1000 == 0:
        print(f"  Checked {count} samples... Found {len(conv_male_samples)} matches so far.")

    if len(conv_male_samples) >= 60:
        break

    if count > 50000:
        print("  Stopping search: scanned 50,000 samples.")
        break

print(f"  Collected {len(conv_male_samples)} CONV Male samples after scanning {count} total entries.")
results_conv = run_evaluation("humair_conv_male", conv_male_samples, text_key="text", limit=60)

‹‹ Loading Conversational Male Samples (Goal: 60) ...


Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

  Checked 1000 samples... Found 0 matches so far.
  Checked 2000 samples... Found 0 matches so far.
  Checked 3000 samples... Found 0 matches so far.
  Checked 4000 samples... Found 0 matches so far.
  Checked 5000 samples... Found 0 matches so far.
  Checked 6000 samples... Found 0 matches so far.
  Checked 7000 samples... Found 0 matches so far.
  Checked 8000 samples... Found 0 matches so far.
  Checked 9000 samples... Found 0 matches so far.
  Checked 10000 samples... Found 0 matches so far.
  Checked 11000 samples... Found 0 matches so far.
  Checked 12000 samples... Found 0 matches so far.
  Collected 60 CONV Male samples after scanning 12900 total entries.
  [01/60] sim=0.5889 gender=Male text=گوا کی یہ تہہ دار میٹھی چیز ہن...
  [02/60] sim=0.6528 gender=Male text=آئیے دیکھتے ہیں کہ حالات ٹھیک ...
  [03/60] sim=0.6282 gender=Male text=میں واقعی اس کی تعریف کرتا ہوں...
  [04/60] sim=0.5800 gender=Male text=اس کے اوقات کیا ہیں؟...
  [05/60] sim=0.6506 gender=Male text=پھر ملتے ہیں

In [ ]:
import zipfile
import os
import pandas as pd

# ── Define domains and their results ────────────────────────────────────────
domains = [
    ("book", results_book, "mushra_humair_book_male"),
    ("wiki", results_wiki, "mushra_humair_wiki_male"),
    ("conv", results_conv, "mushra_humair_conv_male")
]

# ── Generate separate CSVs and ZIPs for each domain ─────────────────────────
for domain_name, results, folder_name in domains:
    # 1. Create and save individual CSV
    df_domain = pd.DataFrame(results)
    df_domain["dataset"] = f"humair_{domain_name}"

    # Select relevant columns
    valid_cols = ["dataset", "sample_id", "urdu_text", "audio_duration_sec", "ref_gen_similarity"]
    df_domain = df_domain[valid_cols]

    csv_filename = f"humair_{domain_name}_evaluation_results.csv"
    df_domain.to_csv(csv_filename, index=False, encoding="utf-8-sig")
    print(f"✅ Saved CSV: {csv_filename}")

    # 2. Create individual ZIP
    zip_filename = f"mushra_humair_{domain_name}.zip"
    if os.path.exists(folder_name):
        with zipfile.ZipFile(zip_filename, "w") as zf:
            for root, _, files in os.walk(folder_name):
                for file in files:
                    zf.write(os.path.join(root, file))
        print(f"✅ Created ZIP: {zip_filename}")
    else:
        print(f"☑ Folder {folder_name} not found, skipping ZIP.")

# ── Summary Display ──────────────────────────────────────────────────────────
all_df = pd.concat([pd.DataFrame(r).assign(dataset=n) for n, r, _ in domains])
stats = all_df.groupby("dataset").agg(
    mean_gen_sim=("ref_gen_similarity", "mean"),
    std_gen_sim=("ref_gen_similarity", "std"),
    count=("sample_id", "count")
)
display(stats)

✅ Saved CSV: humair_book_evaluation_results.csv
✅ Created ZIP: mushra_humair_book.zip
✅ Saved CSV: humair_wiki_evaluation_results.csv
✅ Created ZIP: mushra_humair_wiki.zip
✅ Saved CSV: humair_conv_evaluation_results.csv
✅ Created ZIP: mushra_humair_conv.zip


,mean_gen_sim,std_gen_sim,count
dataset,,,
book,0.603227,0.038089,60
conv,0.578787,0.052742,60
wiki,0.630200,0.038409,30
